In [35]:
import math
import numpy as np 
from pylab import plt, mpl
import scipy
 
# 画图引入中文
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] =False
coupon = 0.0175
par = 100
T = 15
m = 4
bond_yield = 0.0175
price = 100
t_list = np.arange(1,T*m+1)/m
cashflow = np.ones_like(t_list)*coupon*1/m*par
cashflow[-1]= par*(1+coupon*1/m)

print(t_list, cashflow)

[ 0.25  0.5   0.75  1.    1.25  1.5   1.75  2.    2.25  2.5   2.75  3.
  3.25  3.5   3.75  4.    4.25  4.5   4.75  5.    5.25  5.5   5.75  6.
  6.25  6.5   6.75  7.    7.25  7.5   7.75  8.    8.25  8.5   8.75  9.
  9.25  9.5   9.75 10.   10.25 10.5  10.75 11.   11.25 11.5  11.75 12.
 12.25 12.5  12.75 13.   13.25 13.5  13.75 14.   14.25 14.5  14.75 15.  ] [  0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375   0.4375
   0.4375   0.4375   0.4375 100.4375]


In [36]:
# (票面利率、本金、期限、每年coupon支付频次、到期收益率)
def Bond_price (C, M, T, m, y):
    coupon = []
    for i in np.arange(1, T*m+1):
        coupon.append(np.exp(-y*i/m)*M*C/m) # 每一期Coupon折现
    return np.sum(coupon)+np.exp(-y*T)*M    # 每期利息收益折现+到期收益折现



def YTM(C,M,T,m,P):#(票面利率、本金、期限、每年coupon支付频次、价格)
    def f(y):
        coupon= []
        for i in np.arange(1, T*m+1):
            coupon.append(np.exp(-y*i/m)*M*C/m)
        return np.sum(coupon)+np.exp(-y*T)*M-P # 输出一个等于0的公式
    return scipy.optimize.fsolve(f, 0.1) #给定一个随便值求解


# 把未来现金流收回的平均时间
def M_Duration (c, y, t): # 存续期内每期现金流，到期收益率，对应时刻
    cashflow = []
    weight = []
    n = len(t)
    for i in np.arange(n):
        cashflow.append(c[i]*np.exp(-y*t[i]))    ##计算每一期现金流折现
    for i in np.arange(n):
        weight.append(cashflow[i]/sum(cashflow)) #计算每一期现金流与债券价格的比率
    duration = np. sum(t*weight)  #计算麦考利久期
    return duration



def Modi_Duration (c, y, m, t): # 存续期内每期现金流，到期收益率，每年付息几次，对应时刻
    cashflow = []
    weight = []
    n = len(t)
    Rc = m*np.log(1+y/m) # 计算对应的连续复利的债券到期收益率
    for i in np.arange(n):
        cashflow.append(c[i]*np.exp(-Rc*t[i]))    ##计算每一期现金流
    for i in np.arange(n):
        weight.append(cashflow[i]/sum(cashflow)) #计算每一期现金流与债券价格的比率
    duration = np. sum(t*weight)  #计算麦考利久期
    return duration/(1+y) # 计算修正久期



def Convexity (c, y, t): # 现金流、到期收益率连续复利、对应时刻
    cashflow = []
    weight = []
    n = len(t)
    for i in np.arange(n):
        cashflow.append(c[i]*np.exp(-y*t[i]))    ##计算每一期现金流
    for i in np.arange(n):
        weight.append(cashflow[i]/sum(cashflow)) # 计算每一期现金流与债券价格的比率
    convexity = np. sum(weight*t**2)+ np. sum(weight*t)/(1+y)**2# 计算凸性
    return convexity



In [37]:

print(Bond_price(coupon,par,T,m,bond_yield))
print(YTM(coupon, par, T,m,price)[0])
m_duration = M_Duration(cashflow, bond_yield, t_list)
m_bond_price = Bond_price(coupon, par, T, m, bond_yield+0.01)
print(m_duration)

print(Modi_Duration(c=cashflow, y=bond_yield, m=m,t= t_list))
print(Convexity(c=cashflow, y = bond_yield, t= t_list))

99.94953321774123
0.017461830038559927
13.224650748374284
12.997755939870453
202.4936406498792
